# Lab 16 solution: How to choose an agent design

The worked answers to the lab's two challenge tasks, run with their outputs: a lead agent that splits the
five-customer request itself, and the same request grown to ten customers. The setup cells restate what the tasks use
from the lab (the lookups, the requests, the checks, the agents and `measure`), so this notebook runs on its own; the
lab explains each of them.

## Setup

In [1]:
from pathlib import Path
import asyncio
import json
import os
import re
import sqlite3
import time
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


The three lookups on the read-only database, as in Part A:

In [2]:
DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}


for name in ["Helena Holý", "Puja Srivastava"]:
    customer = find_customer(name)[0]
    print(customer, get_invoices(customer["customer_id"]), get_support_rep(customer["customer_id"]))

{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'} {'customer_id': 6, 'invoice_count': 7, 'total': 49.62} {'customer_id': 6, 'support_rep': 'Steve Johnson'}
{'customer_id': 59, 'name': 'Puja Srivastava', 'country': 'India'} {'customer_id': 59, 'invoice_count': 6, 'total': 36.64} {'customer_id': 59, 'support_rep': 'Jane Peacock'}


The requests, the facts a right reply must contain, and the check, as in Part A. `R10` is task 2's request, with
five more customers whose totals come from the database in the next cell.

In [3]:
FIVE = ["Helena Holý", "Puja Srivastava", "Richard Cunningham", "Luis Rojas", "Manoj Pareek"]
MORE = ["Ladislav Kovács", "Frank Ralston", "Astrid Gruber", "Terhi Hämäläinen", "Tim Goyer"]
TEN = FIVE + MORE

REQUESTS = {
    "R3": ("What has each of these five customers spent in total: Helena Holý, Puja Srivastava, "
           "Richard Cunningham, Luis Rojas and Manoj Pareek? Who spent the most?"),
    "R10": "What has each of these ten customers spent in total: " + ", ".join(TEN) + "? Who spent the most?",
}
ANSWERS = {"R3": ["49.62", "36.64", "47.62", "46.62", "38.62"]}
ANSWERS["R10"] = ANSWERS["R3"] + [f"{get_invoices(find_customer(name)[0]['customer_id'])['total']:.2f}" for name in MORE]
print(REQUESTS["R10"])
print(ANSWERS["R10"])


def facts_found(text, facts):
    return [bool(re.search(rf"\b{re.escape(fact)}\b", text or "")) for fact in facts]

What has each of these ten customers spent in total: Helena Holý, Puja Srivastava, Richard Cunningham, Luis Rojas, Manoj Pareek, Ladislav Kovács, Frank Ralston, Astrid Gruber, Terhi Hämäläinen, Tim Goyer? Who spent the most?
['49.62', '36.64', '47.62', '46.62', '38.62', '45.62', '43.62', '42.62', '41.62', '38.62']


Ten totals: the five from the lab, then Ladislav Kovács, Frank Ralston, Astrid Gruber, Terhi Hämäläinen and Tim Goyer.
Tim Goyer's total, $38.62, is the same as Manoj Pareek's, so a reply that drops one of the two still contains the
number: the check cannot see that miss, which is one more reason to read the replies.

The agents from the lab: the desk with all three tools (Part B), the worker and the main agent (Part E).

In [4]:
from agents import Agent, ModelSettings, Runner, function_tool, set_tracing_disabled
from openai.types.shared import Reasoning

set_tracing_disabled(True)
SETTINGS = ModelSettings(reasoning=Reasoning(effort=EFFORT))
FIND, INVOICES, REP = function_tool(find_customer), function_tool(get_invoices), function_tool(get_support_rep)
RULE = "Look up every fact with your tools; never answer from memory. Answer briefly."

desk = Agent(name="Desk", instructions="You help the support reps of a music shop. " + RULE,
             model=MODEL, tools=[FIND, INVOICES, REP], model_settings=SETTINGS)
worker = Agent(name="Customer worker",
               instructions=("You look up one customer's total spend. Look up every fact with your tools; never "
                             "answer from memory. Reply with one line: the name and the total in dollars."),
               model=MODEL, tools=[FIND, INVOICES], model_settings=SETTINGS)
main = Agent(name="Main",
             instructions="You help the support reps of a music shop. Answer from the lines you are given; do not add facts.",
             model=MODEL, model_settings=SETTINGS)
print([agent.name for agent in [desk, worker, main]])

['Desk', 'Customer worker', 'Main']


`numbers_from`, `run_agent`, `measure` and `run_parallel`, as in Parts B and E:

In [5]:
price_in = float(settings["PRICE_INPUT_PER_MILLION"]) / 1_000_000
price_out = float(settings["PRICE_OUTPUT_PER_MILLION"]) / 1_000_000
ROWS = []


def numbers_from(result):
    usage = result.context_wrapper.usage
    return {"requests": usage.requests, "input": usage.input_tokens, "output": usage.output_tokens,
            "final_context": result.raw_responses[-1].usage.input_tokens,
            "turns": [[getattr(part, "name", part.type) for part in r.output] for r in result.raw_responses]}


async def run_agent(agent, question):
    result = await Runner.run(agent, question)
    return result.final_output, numbers_from(result)

In [6]:
async def measure(design, key, run_once, runs=3):
    for n in range(1, runs + 1):
        t0 = time.perf_counter()
        text, numbers = await run_once()
        seconds = time.perf_counter() - t0
        found = facts_found(text, ANSWERS[key])
        cost = numbers["input"] * price_in + numbers["output"] * price_out
        ROWS.append({"design": design, "request": key, "run": n, "text": text, "seconds": seconds,
                     "right": all(found), "cost": cost, **numbers})
        print(f"{design:19} {key:4} run {n} | facts {sum(found)}/{len(found)} {'✅' if all(found) else '❌'} | "
              f"requests {numbers['requests']:2} | input {numbers['input']:5} | "
              f"final context {numbers['final_context']:4} | {seconds:4.1f} s | ${cost:.6f}")


async def run_parallel(names, question):
    worker_runs = await asyncio.gather(*[Runner.run(worker, f"What has {name} spent in total?") for name in names])
    lines = "\n".join(run.final_output for run in worker_runs)
    joined = await Runner.run(main, question + "\n\nWhat the workers found:\n" + lines)
    runs = [*worker_runs, joined]
    return joined.final_output, {
        "requests": sum(run.context_wrapper.usage.requests for run in runs),
        "input": sum(run.context_wrapper.usage.input_tokens for run in runs),
        "output": sum(run.context_wrapper.usage.output_tokens for run in runs),
        "final_context": joined.raw_responses[-1].usage.input_tokens,
        "turns": [],
    }

The two designs from the lab on R3, three runs each, so both tasks compare against numbers from this notebook's own
run:

In [7]:
await measure("one agent", "R3", lambda: run_agent(desk, REQUESTS["R3"]))
await measure("parallel sub-agents", "R3", lambda: run_parallel(FIVE, REQUESTS["R3"]))

one agent           R3   run 1 | facts 5/5 ✅ | requests  3 | input  1460 | final context  752 |  8.5 s | $0.000287


one agent           R3   run 2 | facts 5/5 ✅ | requests  3 | input  1460 | final context  752 |  7.8 s | $0.000287


one agent           R3   run 3 | facts 5/5 ✅ | requests  4 | input  2290 | final context  850 | 10.4 s | $0.000393


parallel sub-agents R3   run 1 | facts 5/5 ✅ | requests 16 | input  3364 | final context  125 | 14.0 s | $0.000507


parallel sub-agents R3   run 2 | facts 5/5 ✅ | requests 16 | input  3364 | final context  125 | 11.2 s | $0.000506


parallel sub-agents R3   run 3 | facts 5/5 ✅ | requests 16 | input  3364 | final context  125 | 11.0 s | $0.000506


The baseline in this run: one agent right in every run at 3 or 4 requests and 1,460 to 2,290 input tokens, with a final context of 752 to 850; parallel sub-agents right at 16 requests and 3,364 tokens, with a final context of 125.

---

## Task 1: Let a lead agent split the work

In the lab our code decided to start five workers. Here a model decides: the worker becomes a tool of a `Lead`
agent, and the lead's instructions say to ask it once per customer, all in one reply. When a model asks for several
agent tools in one reply, the SDK runs them at the same time, so the five workers still run in parallel; the
difference is who wrote the split.

In [8]:
lead = Agent(
    name="Lead",
    instructions=("You help the support reps of a music shop. For a request about several customers, ask the customer "
                  "worker once per customer, all in one reply; never answer from memory. Answer briefly."),
    model=MODEL, model_settings=SETTINGS,
    tools=[worker.as_tool(tool_name="ask_customer_worker",
                          tool_description="Ask a worker for one customer's total spend. Give it one customer's full name.")],
)

await measure("lead + workers", "R3", lambda: run_agent(lead, REQUESTS["R3"]))
for row in ROWS:
    if row["design"] == "lead + workers":
        print(f"run {row['run']}: the lead's requests:", row["turns"])

lead + workers      R3   run 1 | facts 5/5 ✅ | requests 17 | input  3663 | final context  358 | 13.4 s | $0.000591


lead + workers      R3   run 2 | facts 5/5 ✅ | requests 17 | input  3663 | final context  358 | 16.7 s | $0.000587


lead + workers      R3   run 3 | facts 5/5 ✅ | requests 17 | input  3663 | final context  358 | 15.6 s | $0.000587
run 1: the lead's requests: [['ask_customer_worker', 'ask_customer_worker', 'ask_customer_worker', 'ask_customer_worker', 'ask_customer_worker'], ['message']]
run 2: the lead's requests: [['ask_customer_worker', 'ask_customer_worker', 'ask_customer_worker', 'ask_customer_worker', 'ask_customer_worker'], ['message']]
run 3: the lead's requests: [['ask_customer_worker', 'ask_customer_worker', 'ask_customer_worker', 'ask_customer_worker', 'ask_customer_worker'], ['message']]


The lead asked the worker five times in its first reply, one call per customer, and wrote the answer in its second: 2 requests of its own plus five workers of 3, 17 in all, the same in every run. It was right in every run at 3,663 input tokens, a little more than the code split's 3,364, because the lead's own requests carry the tool and its five calls. Its final context is 358 tokens: more than the main agent's 125, since the lead reads its five calls and the five answers, and much less than one agent's 752 to 850. The split now follows the request (a model chose five workers for five names), at the price of one more request and a larger final context.

---

## Task 2: Grow the request to ten customers

The same two designs on ten customers, three runs each:

In [9]:
await measure("one agent", "R10", lambda: run_agent(desk, REQUESTS["R10"]))
await measure("parallel sub-agents", "R10", lambda: run_parallel(TEN, REQUESTS["R10"]))

one agent           R10  run 1 | facts 10/10 ✅ | requests  3 | input  2299 | final context 1278 | 10.0 s | $0.000485


one agent           R10  run 2 | facts 10/10 ✅ | requests  3 | input  2299 | final context 1278 | 10.3 s | $0.000485


one agent           R10  run 3 | facts 10/10 ✅ | requests  3 | input  2299 | final context 1278 |  9.5 s | $0.000485


parallel sub-agents R10  run 1 | facts 10/10 ✅ | requests 31 | input  6713 | final context  205 | 13.4 s | $0.001008


parallel sub-agents R10  run 2 | facts 10/10 ✅ | requests 31 | input  6713 | final context  205 | 13.9 s | $0.001008


parallel sub-agents R10  run 3 | facts 10/10 ✅ | requests 31 | input  6713 | final context  205 | 14.7 s | $0.001008


Every reply that missed a fact, in full, then the comparison with R3:

In [10]:
for row in ROWS:
    if not row["right"]:
        print(f"{row['design']}, {row['request']} run {row['run']}: {row['text']}")
print()


def span(rows, field, fmt="{}"):
    low = fmt.format(min(row[field] for row in rows))
    high = fmt.format(max(row[field] for row in rows))
    return low if low == high else f"{low}-{high}"


print(f"{'request':7} {'design':19} {'right':>5} {'requests':>8} {'input tokens':>12} {'final context':>13} {'seconds':>9}")
for key in ["R3", "R10"]:
    for design in ["one agent", "lead + workers", "parallel sub-agents"]:
        rows = [row for row in ROWS if row["design"] == design and row["request"] == key]
        if rows:
            right = f"{sum(row['right'] for row in rows)}/{len(rows)}"
            print(f"{key:7} {design:19} {right:>5} {span(rows, 'requests'):>8} {span(rows, 'input'):>12} "
                  f"{span(rows, 'final_context'):>13} {span(rows, 'seconds', '{:.1f}'):>9}")


request design              right requests input tokens final context   seconds
R3      one agent             3/3      3-4    1460-2290       752-850  7.8-10.4
R3      lead + workers        3/3       17         3663           358 13.4-16.7
R3      parallel sub-agents   3/3       16         3364           125 11.0-14.0
R10     one agent             3/3        3         2299          1278  9.5-10.3
R10     parallel sub-agents   3/3       31         6713           205 13.4-14.7


Every reply had all ten totals. From five customers to ten, one agent's final context went from 752 (850 in one run) to 1,278 tokens, about 105 more per customer, and the main agent's from 125 to 205, 16 more per customer. In total tokens one agent stayed cheaper, 2,299 against 6,713, and took 3 requests every time (all ten customers in one reply, all ten totals in the next); the parallel design grew with the work, to 31 requests. One agent's last request passed 1,024 tokens, where prompt caching can start; `measure` prices every input token at the full price, so its cost here is an upper bound.

**What the two tasks show.** Who splits the work (our code or a lead model) changes the request count by one and the final context by the size of the lead's tool calls. The number of customers changes how fast one agent's context grows. On this desk's small lookups one agent stays the cheapest in total; parallel sub-agents keep the answering agent's context nearly flat as the request grows, which pays when each part reads far more than it returns.